# Python for Health, Economic, and Social Science
## Lab 6 (Expansion): Local LLMs with Ollama

**Course:** LCDS, Nuffield College Oxford  
**Status:** Optional expansion — for participants who want to explore Large Language Models in research workflows

---

### What this lab covers

Ollama lets you run open-weight LLMs (Llama, Mistral, Gemma, etc.) **entirely on your own machine** — no API key, no data leaving your laptop, no cost per token. This makes it suitable for:

- Research with sensitive data that cannot go to cloud APIs
- Exploring LLM capabilities without cost commitments
- Teaching and prototyping before committing to a paid API

**By the end of this lab you will be able to:**
1. Install Ollama and pull a model
2. Send prompts and receive completions from Python
3. Use structured output (JSON mode) to extract fields from text
4. Build a simple retrieval-augmented generation (RAG) pattern
5. Score and classify short social-care notes using a local LLM

---

### Requirements

| Requirement | Notes |
|-------------|-------|
| Ollama installed | https://ollama.com/download (macOS/Linux/Windows) |
| `ollama` Python package | `pip install ollama` |
| Model pulled | `ollama pull llama3.2` (≈2 GB) or `ollama pull mistral` (≈4 GB) |
| RAM | ≥8 GB recommended; 4 GB usable with very small models |

The model runs locally — first pull takes a few minutes but subsequent runs are instant.

---
## Part 0: Setup and Installation

In [ ]:
# Run this cell once to install the Ollama Python client
# (the Ollama desktop app must already be installed separately from https://ollama.com)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "ollama", "-q"])
print("ollama package installed")

In [ ]:
# Check Ollama server is running and print available local models
import ollama

try:
    models = ollama.list()
    for m in models.models:
        print(m.model, "|", round(m.size / 1e9, 1), "GB")
except Exception as e:
    print(f"Ollama server not reachable: {e}")
    print("Make sure the Ollama application is running (system tray / menu bar icon).")

In [ ]:
# Choose a model — adjust based on what you have pulled
# Alternatives: 'mistral', 'gemma3:4b', 'phi3:mini', 'qwen2.5:3b'
MODEL = "llama3.2"

# Quick test — if this cell runs, everything is working
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly three words: Python is"}]
)
print(response.message.content)

> **Note on model choice:**
> - `llama3.2` (3B): fast, fits on 8 GB RAM, good for structured tasks
> - `mistral` (7B): better reasoning, requires ≥16 GB RAM
> - `phi3:mini` (3.8B): excellent instruction-following, very small
> - `gemma3:4b`: Google's model, strong at text classification
>
> For all exercises in this lab, `llama3.2` or `phi3:mini` is sufficient.

---
## Part 1: Basic Chat

### Exercise 1.1: Single-turn prompt

In [ ]:
# Simple single-turn chat
def ask(prompt: str, model: str = MODEL) -> str:
    """Send a user prompt, return the model's text reply."""
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.message.content

reply = ask("In two sentences, explain what a Gini coefficient measures.")
print(reply)

### Exercise 1.2: System prompts — controlling persona and output format

In [ ]:
# System prompt sets the model's role/behaviour for the whole conversation
def ask_with_system(system: str, prompt: str, model: str = MODEL) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ]
    )
    return response.message.content

SYSTEM_POLICY = (
    "You are a public health policy analyst for a UK local authority. "
    "Reply in plain English, no jargon, max 3 bullet points."
)

reply = ask_with_system(
    SYSTEM_POLICY,
    "What are the main drivers of emergency hospital admissions in deprived urban areas?"
)
print(reply)

> **Key concept — roles in the chat API:**
> - `system` — invisible instruction to the model (sets tone, format, constraints)
> - `user` — the human's message
> - `assistant` — the model's previous reply (used for multi-turn conversations)
>
> The system prompt is one of the most powerful tools for making LLM output predictable and useful in research pipelines.

### Exercise 1.3: Multi-turn conversation

In [ ]:
# Multi-turn: build a message history
messages = [
    {"role": "system", "content": "You are a helpful statistics tutor. Keep answers brief."},
    {"role": "user",   "content": "What is OLS regression?"},
]

resp1 = ollama.chat(model=MODEL, messages=messages)
messages.append({"role": "assistant", "content": resp1.message.content})
print("A:", resp1.message.content)

# Follow-up question (model has context from previous turn)
messages.append({"role": "user", "content": "And what is its key assumption about residuals?"})
resp2 = ollama.chat(model=MODEL, messages=messages)
print("\nA:", resp2.message.content)

---
## Part 2: Structured JSON Output

Ollama supports JSON mode — the model returns valid JSON instead of free text. This makes LLM output directly parseable as Python objects.

### Exercise 2.1: Extract structured fields from a case note

In [ ]:
import json

def extract_json(note: str, model: str = MODEL) -> dict:
    """Ask the model to parse a case note and return structured JSON."""
    prompt = f"""Extract structured information from the following case note.
Return ONLY valid JSON with these keys:
  - risk_factors: list of strings
  - urgency: one of ["immediate", "within_week", "routine"]
  - suggested_referral: string (service or agency name)
  - summary: one sentence

Case note: {note}"""

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        format="json"      # forces JSON output mode
    )
    return json.loads(response.message.content)


notes = [
    "Patient missed two appointments, reports housing arrears and recent welfare sanction.",
    "Elderly resident isolated after bereavement; chronic condition deteriorating, expressing hopelessness.",
    "No concerns; employed, stable housing, family support network in place.",
]

for note in notes:
    result = extract_json(note)
    print(f"--- NOTE: {note[:60]}...")
    print(json.dumps(result, indent=2))
    print()

> **Why JSON mode matters for research:** You can now turn unstructured text into a pandas DataFrame of structured fields — no regex, no manual labelling. This is the foundation of LLM-assisted data extraction pipelines used in systematic reviews, social media analysis, and clinical note processing.

### Exercise 2.2: Batch processing — build a triage DataFrame from case notes

In [ ]:
import pandas as pd

case_notes_batch = [
    "Patient missed two appointments this month due to transport issues and housing arrears.",
    "Carer with anxiety after welfare sanction; family income severely impacted.",
    "A&E attendance for chest pain; unemployed, reports severe financial stress.",
    "School absence elevated; child with asthma, parent seeking benefits advice.",
    "Patient engaged with preventive programme, income stable, low risk.",
    "Recurring missed medications; cannot afford prescriptions on universal credit.",
    "Social worker referral: debt and food insecurity in high deprivation area.",
    "No concerns; regular attendance, employed, supportive family network.",
]

records = []
for i, note in enumerate(case_notes_batch):
    try:
        parsed = extract_json(note)
        records.append({
            "note_id":          i + 1,
            "urgency":          parsed.get("urgency", "unknown"),
            "suggested_referral": parsed.get("suggested_referral", ""),
            "n_risk_factors":   len(parsed.get("risk_factors", [])),
            "summary":          parsed.get("summary", ""),
        })
    except (json.JSONDecodeError, Exception) as e:
        print(f"Note {i+1} failed: {e}")
        records.append({"note_id": i+1, "urgency": "error"})

triage_llm = pd.DataFrame(records)
print(triage_llm.sort_values("urgency"))

> **Instructor note:** Production systems wrap LLM calls in retry logic and parse failures gracefully — models occasionally produce malformed JSON. The `try/except` is not optional here.
>
> **Expansion — using Pydantic for typed structured output:**
```python
# pip install pydantic
from pydantic import BaseModel
from typing import Literal

class CaseNoteSchema(BaseModel):
    risk_factors:       list[str]
    urgency:            Literal["immediate", "within_week", "routine"]
    suggested_referral: str
    summary:            str

raw = extract_json(case_notes_batch[0])
validated = CaseNoteSchema(**raw)   # raises ValidationError if schema violated
print(validated.urgency)
```

---
## Part 3: Retrieval-Augmented Generation (RAG) — Light Pattern

RAG = give the model relevant context from your own documents, then ask questions about it.  
The simplest version: paste the document (or relevant chunks) directly into the prompt.

### Exercise 3.1: Question-answering over a policy document

In [ ]:
# Simulated policy document (replace with a real file in practice)
POLICY_DOC = """
NORTHSIDE LOCAL AUTHORITY — PUBLIC HEALTH STRATEGY 2024-2026

1. PRIMARY CARE INVESTMENT
The authority will invest £2.4 million over two years to expand GP capacity in the
Northside and Riverside wards, focusing on preventive care, chronic disease management,
and reducing unnecessary emergency admissions.

2. SOCIAL PRESCRIBING
A network of 12 Social Prescribing Link Workers will be embedded in primary care teams
by Q3 2024. Referrals will target patients with social risk factors including financial
insecurity, isolation, and housing instability.

3. SCHOOL ABSENCE REDUCTION
A joint initiative with the education authority targets a 15% reduction in persistent
absenteeism by improving family access to GP services and mental health support for
pupils in Years 7-9.

4. DATA AND OUTCOMES
Progress will be monitored against three key indicators: emergency admission rates
per 1,000 population; school absence rates; and uptake of preventive screenings.
Quarterly performance reports will be published on the council website.
"""

def ask_about_document(document: str, question: str, model: str = MODEL) -> str:
    prompt = f"""The following is a policy document:

---
{document}
---

Based only on the document above, answer this question concisely:
{question}"""
    return ask(prompt, model)


questions = [
    "What is the total investment in primary care and over what period?",
    "How many Social Prescribing Link Workers will be deployed?",
    "What are the three key outcome indicators?",
    "Is there anything in the document about mental health services for adults?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_about_document(POLICY_DOC, q)}\n")

> **Note on the last question:** A well-aligned model should say there is **no information** about adult mental health in the document, rather than hallucinating. If it does hallucinate, this is a prompt engineering failure — add the instruction "if the answer is not in the document, say 'Not mentioned'"; to the prompt.
>
> **Expansion — loading a real PDF or text file:**
```python
# pip install pypdf2
import PyPDF2

def load_pdf_text(filepath: str) -> str:
    text = ""
    with open(filepath, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

doc_text = load_pdf_text("../Data/policy_report.pdf")
reply = ask_about_document(doc_text[:4000], "What are the key recommendations?", MODEL)
print(reply)
# Note: truncate to context window (typically 4k-128k tokens depending on model)
```

### Exercise 3.2: Summarise a collection of documents

In [ ]:
# Summarise multiple short documents (simulate reading multiple reports)
report_excerpts = [
    "Riverside ward recorded 158 emergency admissions per 1,000 in Q4 2023, up 12% year-on-year.",
    "Social prescribing referrals in Northside increased by 37% in 2023-24, with positive outcomes reported.",
    "School absence rates in Moorfield reached 10.3%, highest across the authority.",
    "Preventive screening uptake remains below national target in all rural wards.",
]

combined = "\n".join(f"- {r}" for r in report_excerpts)
prompt = f"""You are a policy analyst. Summarise the following findings in 3 concise bullet points
identifying the highest-priority action areas:\n\n{combined}"""

print(ask(prompt))

---
## Part 4: Demographic and Economic Classification

### Exercise 4.1: Classify area risk from tabular data

In [ ]:
# Represent a row of data as text, then ask the LLM to classify it
import pandas as pd

areas_df = pd.DataFrame([
    {"area": "Northside",  "unemployment": 11.8, "deprivation": 72, "absence": 8.2, "income": 29000},
    {"area": "Central",    "unemployment": 7.4,  "deprivation": 31, "absence": 5.1, "income": 37000},
    {"area": "Moorfield",  "unemployment": 15.2, "deprivation": 85, "absence": 10.3,"income": 22000},
    {"area": "Southdowns", "unemployment": 5.6,  "deprivation": 18, "absence": 4.1, "income": 42000},
])

def classify_area(row: pd.Series, model: str = MODEL) -> str:
    prompt = f"""Based on the following health and economic indicators for a UK local area,
classify it as HIGH, MODERATE, or LOW priority for public health investment.
Give a one-sentence rationale.

Area: {row['area']}
Unemployment rate: {row['unemployment']}%
Deprivation index: {row['deprivation']} (scale 0-100)
School absence rate: {row['absence']}%
Median income: £{row['income']:,}

Reply in JSON with keys: priority (HIGH/MODERATE/LOW), rationale (string)"""

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        format="json"
    )
    try:
        return json.loads(response.message.content)
    except json.JSONDecodeError:
        return {"priority": "ERROR", "rationale": response.message.content[:100]}


results = []
for _, row in areas_df.iterrows():
    result = classify_area(row)
    results.append({"area": row["area"], **result})

print(pd.DataFrame(results))

> **Research application:** This pattern — convert tabular row to text, ask LLM for classification + explanation, collect in DataFrame — provides **LLM-generated qualitative labels** for quantitative records. It can augment dataset annotation, literature screening, or rapid policy profiling.

---
## Part 5: Streaming and Performance

### Exercise 5.1: Streaming output (better UX for long responses)

In [ ]:
# Streaming: print tokens as they arrive (no waiting for full response)
stream = ollama.chat(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": "Write a 5-line summary of what causes health inequalities in urban areas."
    }],
    stream=True
)

for chunk in stream:
    print(chunk.message.content, end='', flush=True)
print()  # newline at end

### Exercise 5.2: Measure response time

In [ ]:
import time

def timed_ask(prompt: str, model: str = MODEL) -> tuple[str, float]:
    start = time.time()
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    return response.message.content, elapsed

reply, t = timed_ask("Name three UK government datasets relevant to public health research.")
print(reply)
print(f"\nResponse time: {t:.1f}s")

---
## Part 6: Practical Patterns for Research Use

### Exercise 6.1: Prompt templates for research tasks

In [ ]:
# Reusable prompt templates — promote consistency across many calls
TEMPLATES = {
    "summarise": lambda text: f"Summarise the following in 3 bullet points:\n\n{text}",

    "extract_terms": lambda text: (
        f"Extract all named clinical conditions, social risk factors, and policy interventions "
        f"from this text. Return as JSON with keys: conditions, risk_factors, interventions.\n\n{text}"
    ),

    "compare": lambda a, b: (
        f"Compare the following two areas based on their descriptions. "
        f"Return a brief comparison in 2 sentences.\n\nArea A: {a}\nArea B: {b}"
    ),

    "translate_jargon": lambda text: (
        f"Rewrite the following public health text in plain English for a general audience "
        f"(no technical jargon):\n\n{text}"
    ),
}

# Example: extract clinical terms from a note
note = """Patient presents with COPD exacerbation and comorbid T2DM.
Social circumstances: benefits sanctions have led to food insecurity.
Referred to social prescribing link worker."""

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": TEMPLATES["extract_terms"](note)}],
    format="json"
)
print(json.dumps(json.loads(response.message.content), indent=2))

### Exercise 6.2: Evaluating LLM output against a gold standard

In [ ]:
# Simulate evaluation: compare LLM classifications to hand-labelled labels
gold_standard = {
    "Moorfield":  "HIGH",
    "Northside":  "HIGH",
    "Central":    "MODERATE",
    "Southdowns": "LOW"
}

# (Use results from Exercise 4.1)
if 'results' in dir():
    eval_df = pd.DataFrame(results)
    eval_df["gold"] = eval_df["area"].map(gold_standard)
    eval_df["correct"] = eval_df["priority"] == eval_df["gold"]
    print(eval_df[["area", "priority", "gold", "correct"]])
    print(f"\nAccuracy: {eval_df['correct'].mean():.0%}")
else:
    print("Run Exercise 4.1 first to generate results.")

> **Why evaluation matters:** LLMs are probabilistic — they may give different answers on different runs. Before using LLM-generated labels in a research pipeline, always validate against a hand-labelled sample. A confusion matrix and inter-rater reliability metric (Cohen's Kappa) should accompany any LLM annotation methodology section.

---
## Summary: When to Use a Local LLM

| Task | Recommendation |
|------|----------------|
| Batch classification of notes/text | ✅ Local LLM (privacy, cost) |
| Structured data extraction from free text | ✅ JSON mode |
| Q&A over a specific document | ✅ RAG pattern |
| Tabular data modelling (regression, classification) | ❌ Use statsmodels/sklearn |
| High-accuracy classification with labelled data | ❌ Fine-tuned model or sklearn |
| Sensitive data with governance constraints | ✅ Local LLM only |
| Explaining results to non-technical audiences | ✅ LLM can help draft |

**Further reading:**
- [Ollama documentation](https://ollama.com/docs)
- [Ollama Python library](https://github.com/ollama/ollama-python)
- LangChain / LlamaIndex for production RAG pipelines
- OpenAI API for cloud-based equivalents (same interface pattern)

---
*LCDS Oxford Python Course, March 2026 — Lab 6 (Expansion: Ollama / Local LLMs)*